# Qiskit Fluency Check

## Objectives
- Construct basic quantum circuits in Qiskit.
- Demonstrate core gates: H, CX and verify their effects.
- Verify state preparation using analytical tools (Statevector/DensityMatrix).
- Sample measurement outcomes on an ideal simulator (Aer).
- Visualize single-qubit states on the Bloch sphere.

## Setup


In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer.primitives import Sampler 
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector

## Theory Snapshot
- **H gate** creates superposition; maps $|0\rangle \rightarrow (|0\rangle+|1\rangle)/\sqrt{2}, \quad |1\rangle \rightarrow (|0\rangle−|1\rangle)/\sqrt{2}$.
- **CX gate** entangles control and target, conditionally flipping the target.
- **Statevector** simulates ideal, noiseless evolution and allows exact state inspection.
- **Aer simulator + measurement** provides empirical counts consistent with Born's rule.

## Demonstration / Example Circuit
Integrated demo: single-qubit, Bell, parameterized circuit, teleportation.

### 1) Single-qubit prep & Bloch

In [ ]:
qc=QuantumCircuit(1); qc.h(0)
sv=Statevector.from_instruction(qc)
display(qc.draw('mpl'))
display(plot_bloch_multivector(sv))

### 2) Bell + basis measurements

In [ ]:
bell=QuantumCircuit(2); bell.h(0); bell.cx(0,1)
z=bell.copy(); z.measure_all()
x=bell.copy(); x.h([0,1]); x.measure_all()
y=bell.copy(); y.sdg([0,1]); y.h([0,1]); y.measure_all()
sampler = Sampler()
rz=sampler.run(z,shots=1024).result().quasi_dists[0]
rx=sampler.run(x,shots=1024).result().quasi_dists[0]
ry=sampler.run(y,shots=1024).result().quasi_dists[0]
plot_histogram([rz,rx,ry], legend=['Z','X','Y'])

### 3) Parameterized 2-qubit circuit

In [ ]:
theta=Parameter('θ'); qp=QuantumCircuit(2)
qp.ry(theta,0); qp.cx(0,1); qp.rx(theta,1); qp.measure_all()
sim=Aer.get_backend('aer_simulator')
plot_histogram(sim.run(qp.assign_parameters({theta:0.7}), shots=1024).result().get_counts())

### 4) Teleportation (compact)

In [ ]:
tel=QuantumCircuit(3,3)
# Entangle 1-2
tel.h(1); tel.cx(1,2)
# Input |+> on 0
tel.h(0)
# Bell meas
tel.cx(0,1); tel.h(0)
# Correct qubit 2
tel.cx(1,2); tel.cz(0,2) # after classical bits 0, 1 have been communicated
# Verify
tel.measure(2,2)
display(tel.draw('mpl'))
plot_histogram(sim.run(tel, shots=1024).result().get_counts())

## Results & Discussion

### Results
- **Code 1:** $H|0\rangle \rightarrow |+\rangle$, so the Bloch vector correctly lies on the +X axis on the equator.
- **Code 2:** Z/X bases show correlated outcomes (00/11 dominant); Y shows anti-correlation (01/10 more likely).
- **Code 3:** The histogram is heavily biased toward 00, with small leakage into the other three states, matching the expected near-$∣00\rangle$ state.
- **Code 4:** With Z-basis on the receiver, counts are $\approx 50/50$ for $∣+\rangle$.

### Discussion
- **Code 1:** Placement at +X confirms correct relative phase (0) and amplitudes.
- **Code 2:** Patterns match stabilizers of $(\langle ZZ\rangle=+1, \langle XX\rangle=+1, \langle YY\rangle=−1)$ within shot noise.
- **Code 3:** The slight population in non-zero states is consistent with finite-shot noise, while the strong 00 dominance confirms correct circuit preparation.
- **Code 4:** Classical-conditioned corrections reproduce the input on qubit-2; uniformity in the control bits (m₀,m₁) is expected.